# SAR + Optik Fuzyon — Kaggle Eğitim Şablonu

**Kullanım:**
1. Kaggle'da yeni bir Notebook aç (GPU T4 / P100 seçili)
2. Sağdaki `+ Add data` ile M4-SAR datasetini ekle (Kaggle'a yüklü olmalı)
3. Bu notebook'u kopyala ve hücreleri sırayla çalıştır

**Beklenen süre:**
- 1 epoch: ~7-10 dakika (T4 GPU, batch=16, 640×640)
- 80 epoch: ~10 saat (Kaggle haftalık 30 saat sınırı içinde)

## 1. Repoyu klonla ve bağımlılıkları kur

In [ ]:
!git clone https://github.com/roham/sar-optical-fusion.git
%cd sar-optical-fusion
!pip install -q -r requirements.txt
!pip install -e .

## 2. Veri yolu kontrol et

In [ ]:
import os
# Kaggle dataset yolu — kendi yüklediğin path'e göre güncelle
M4_SAR_PATH = '/kaggle/input/m4-sar/m4_sar'
assert os.path.exists(M4_SAR_PATH), f'Veri yolu bulunamadı: {M4_SAR_PATH}'
print('Optik klasörü:', os.listdir(os.path.join(M4_SAR_PATH, 'optical', 'train'))[:3])
print('SAR klasörü:', os.listdir(os.path.join(M4_SAR_PATH, 'sar', 'train'))[:3])

## 3. Konfigürasyonu Kaggle yoluna uyarla

In [ ]:
import yaml
with open('configs/multimodal_full.yaml', 'r') as f:
    cfg = yaml.safe_load(f)
cfg['data']['data_root'] = M4_SAR_PATH
cfg['logging']['output_dir'] = '/kaggle/working/runs'
cfg['training']['batch_size'] = 16   # T4 için uygun
with open('configs/kaggle.yaml', 'w') as f:
    yaml.dump(cfg, f)
print('Kaggle config kaydedildi.')

## 4. Sanity test

In [ ]:
!pytest tests/ -v --tb=short -x

## 5. Eğitim — kısa deneme (3 epoch)

In [ ]:
!python -m src.train --config configs/kaggle.yaml --epochs 3

## 6. Tam eğitim (80 epoch)

In [ ]:
# Uyarı: bu hücre ~10 saat sürer. Kaggle GPU oturumu 12 saatte kapanır.
!python -m src.train --config configs/kaggle.yaml --epochs 80

## 7. Değerlendirme

In [ ]:
!python -m src.eval \
    --checkpoint /kaggle/working/runs/final.pt \
    --config configs/kaggle.yaml

## 8. Çıktıları indir

Kaggle Notebook'un sağ üst köşesinde `Output` sekmesinden tüm `.pt` dosyalarını indir.
Veya `/kaggle/working/` klasörünü zip'leyip indir:

In [ ]:
!cd /kaggle/working && tar czf runs.tar.gz runs/
from IPython.display import FileLink
FileLink('/kaggle/working/runs.tar.gz')